In [1]:
from ultralytics import YOLO

data_dir = r"N:\alpenv\GOES\Projects\Bugnet\BugNet Davos\Students\Carole\Machine Learnig Models\classifier"  # folder that contains train/ and val/

In [ ]:
model = YOLO("yolov8n-cls.pt")  # n = nano (smallest)

model.train(
    data=data_dir,
    epochs=30,       # start small, you can increase later
    imgsz=224,       # default for classification
    batch=30,        # reduce if you run out of RAM
    patience=5,      # early stopping
)


In [ ]:
from ultralytics import YOLO
import os
import shutil

# path to the trained classifier
cls_model_path = r"N:\alpenv\GOES\Projects\Bugnet\BugNet Davos\Students\Carole\Machine Learning Models\classifier\classifier_best.pt"
model = YOLO(cls_model_path)

source_dir = r"N:\alpenv\GOES\Projects\Bugnet\BugNet Davos\Students\Carole\Machine Learning Models\crops_to_sort\output"
dest_root  = r"N:\alpenv\GOES\Projects\Bugnet\BugNet Davos\Students\Carole\Machine Learning Models\crops_sorted"

os.makedirs(dest_root, exist_ok=True)

# class names from your trained model (e.g. {'0': 'class1', ...} or {0: 'class1', ...})
class_names = model.names

# make subfolders for each class
for cname in class_names.values():
    os.makedirs(os.path.join(dest_root, cname), exist_ok=True)

for fname in os.listdir(source_dir):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    fpath = os.path.join(source_dir, fname)
    results = model.predict(fpath, verbose=False)

    probs = results[0].probs
    top_idx = int(probs.top1)
    top_conf = float(probs.top1conf)
    top_name = class_names[top_idx]

    # Only auto-move if confident enough
    if top_conf < 0.85:
        print(f"Low conf {top_conf:.2f} for {fname} -> {top_name}, skipping.")
        continue

    dst = os.path.join(dest_root, top_name, fname)
    shutil.move(fpath, dst)
